# Comparativa de circuitos variacionales para un QAE

Compara cinco ansätze base y dos variantes algo más expresivas. La métrica principal es la fidelidad del trash: la probabilidad de encontrar todos los qubits descartados en `|0>`. Como complemento, se proyecta el trash a `|0>` y se aplica el circuito inverso para obtener una reconstrucción aproximada y su MSE sobre probabilidades.

> El entrenamiento usa una muestra pequeña por circuito para que el notebook sea razonable en simulación. Aumenta `N_TRAIN` y `N_EVAL` para los resultados finales.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

SEED, TOTAL_QUBITS, LATENT_QUBITS = 42, 7, 6
TRASH_QUBITS = TOTAL_QUBITS - LATENT_QUBITS
N_LAYERS, N_TRAIN, N_EVAL, N_EPOCHS, BATCH_SIZE = 2, 24, 16, 12, 8
LEARNING_RATE = 0.08
rng = np.random.default_rng(SEED)

# Reutiliza el pipeline existente si X_train_q/X_val_q/X_test_q ya están en memoria.
if not all(name in globals() for name in ('X_train_q', 'X_val_q', 'X_test_q')):
    candidates = ['../paderborn_features_tratadas.csv', 'paderborn_features_tratadas.csv']
    csv_path = next((p for p in candidates if os.path.exists(p)), None)
    if csv_path is None:
        raise FileNotFoundError('No se encontró paderborn_features_tratadas.csv.')
    df = pd.read_csv(csv_path)
    label = 'Label' if 'Label' in df else None
    X = df.drop(columns=[label] if label else []).replace([np.inf, -np.inf], np.nan).dropna().to_numpy(float)
    X_train, X_tmp = train_test_split(X, test_size=0.30, random_state=SEED)
    X_val, X_test = train_test_split(X_tmp, test_size=0.50, random_state=SEED)
    scaler = RobustScaler().fit(X_train)
    def encode_rows(values):
        scaled = scaler.transform(values)
        size = 2**TOTAL_QUBITS
        padded = np.pad(scaled, ((0, 0), (0, max(0, size - scaled.shape[1]))))[:, :size]
        norms = np.linalg.norm(padded, axis=1, keepdims=True)
        return padded / np.where(norms == 0, 1.0, norms)
    X_train_q, X_val_q, X_test_q = map(encode_rows, (X_train, X_val, X_test))

X_train_q, X_val_q, X_test_q = [np.asarray(x, dtype=float) for x in (X_train_q, X_val_q, X_test_q)]
if X_train_q.shape[1] != 2**TOTAL_QUBITS:
    raise ValueError(f'Cada estado debe tener {2**TOTAL_QUBITS} amplitudes.')
train_data, eval_data = X_train_q[:N_TRAIN], X_val_q[:N_EVAL]
print(f'Train: {len(train_data)} | Evaluación: {len(eval_data)} | Qubits: {TOTAL_QUBITS}')

## Ansätze comparados

Los cinco primeros corresponden a la propuesta original. `EfficientSU2` y `Brickwork RY+RZ` añaden expresividad sin introducir técnicas difíciles de explicar: el primero usa rotaciones completas `Rot` y el segundo alterna las parejas de CNOT.

In [ ]:
def basic_entangler(weights, wires):
    for layer in weights:
        for i, wire in enumerate(wires): qml.RX(layer[i, 0], wires=wire)
        for a, b in zip(wires[:-1], wires[1:]): qml.CNOT(wires=[a, b])

def strongly_entangling(weights, wires):
    for layer in weights:
        for i, wire in enumerate(wires): qml.Rot(*layer[i], wires=wire)
        for i, wire in enumerate(wires): qml.CNOT(wires=[wire, wires[(i + 1) % len(wires)]])

def ry_rz_circular(weights, wires):
    for layer in weights:
        for i, wire in enumerate(wires):
            qml.RY(layer[i, 0], wires=wire); qml.RZ(layer[i, 1], wires=wire)
        for a, b in zip(wires[:-1], wires[1:]): qml.CNOT(wires=[a, b])
        if len(wires) > 2: qml.CNOT(wires=[wires[-1], wires[0]])

def hardware_efficient(weights, wires):
    for layer in weights:
        for i, wire in enumerate(wires):
            qml.RY(layer[i, 0], wires=wire); qml.RZ(layer[i, 1], wires=wire)
        for a, b in zip(wires[:-1], wires[1:]): qml.CZ(wires=[a, b])

def simplified_two_design(weights, wires):
    for layer_id, layer in enumerate(weights):
        for i, wire in enumerate(wires): qml.RY(layer[i, 0], wires=wire)
        for i in range(layer_id % 2, len(wires) - 1, 2): qml.CNOT(wires=[wires[i], wires[i + 1]])

def efficient_su2(weights, wires):
    for layer in weights:
        for i, wire in enumerate(wires): qml.Rot(*layer[i], wires=wire)
        for a, b in zip(wires[:-1], wires[1:]): qml.CNOT(wires=[a, b])

def brickwork_rot(weights, wires):
    for layer_id, layer in enumerate(weights):
        for i, wire in enumerate(wires):
            qml.RY(layer[i, 0], wires=wire); qml.RZ(layer[i, 1], wires=wire)
        for i in range(layer_id % 2, len(wires) - 1, 2): qml.CNOT(wires=[wires[i], wires[i + 1]])

ANSATZE = {
    'BasicEntanglerLayers': (basic_entangler, 2),
    'StronglyEntanglingLayers': (strongly_entangling, 3),
    'RY+RZ Circular': (ry_rz_circular, 2),
    'Hardware-Efficient CZ': (hardware_efficient, 2),
    'Simplified Two-Design': (simplified_two_design, 1),
    'EfficientSU2': (efficient_su2, 3),
    'Brickwork RY+RZ': (brickwork_rot, 2),
}
WIRES = list(range(TOTAL_QUBITS))
trash_wires = list(range(LATENT_QUBITS, TOTAL_QUBITS))

## Métricas de complejidad

La profundidad se obtiene con `qml.specs`; el conteo de puertas se separa en puertas de uno y dos qubits.

In [ ]:
def calcular_metricas_circuito(ansatz, param_count):
    dev = qml.device('default.qubit', wires=TOTAL_QUBITS)
    @qml.qnode(dev)
    def probe(weights):
        qml.AmplitudeEmbedding(train_data[0], wires=WIRES, normalize=False)
        ansatz(weights, WIRES)
        return qml.expval(qml.PauliZ(0))
    weights = pnp.zeros((N_LAYERS, TOTAL_QUBITS, param_count))
    with qml.tape.QuantumTape() as tape:
        qml.AmplitudeEmbedding(train_data[0], wires=WIRES, normalize=False)
        ansatz(weights, WIRES)
    resources = qml.specs(probe)(weights).get('resources', {})
    depth = resources.depth if hasattr(resources, 'depth') else resources.get('depth', np.nan)
    one = sum(len(op.wires) == 1 for op in tape.operations)
    two = sum(len(op.wires) == 2 for op in tape.operations)
    return {'depth': int(depth), 'width': TOTAL_QUBITS, 'parameters': weights.size,
            'one_qubit_gates': one, 'two_qubit_gates': two, 'total_gates': one + two}

complexity = pd.DataFrame({name: calcular_metricas_circuito(fn, n) for name, (fn, n) in ANSATZE.items()}).T
display(complexity)
complexity[['depth', 'width', 'parameters']].plot(kind='bar', figsize=(13, 5), rot=35,
                                                   title='Profundidad, ancho y parámetros')
plt.ylabel('Cantidad'); plt.tight_layout(); plt.show()

## Entrenamiento reutilizable

Se optimiza `1 - F`, donde `F` es la probabilidad de `|0...0>` en los qubits trash. Cada circuito usa la misma muestra, optimizador y número de épocas.

In [ ]:
def crear_qnodes(ansatz):
    dev = qml.device('default.qubit', wires=TOTAL_QUBITS)
    @qml.qnode(dev, interface='autograd')
    def fidelity_qnode(state, weights):
        qml.AmplitudeEmbedding(state, wires=WIRES, normalize=False)
        ansatz(weights, WIRES)
        return qml.probs(wires=trash_wires)
    @qml.qnode(dev)
    def encoded_state(state, weights):
        qml.AmplitudeEmbedding(state, wires=WIRES, normalize=False)
        ansatz(weights, WIRES)
        return qml.state()
    @qml.qnode(dev)
    def decoded_probs(projected_state, weights):
        qml.AmplitudeEmbedding(projected_state, wires=WIRES, normalize=False)
        qml.adjoint(ansatz)(weights, WIRES)
        return qml.probs(wires=WIRES)
    return fidelity_qnode, encoded_state, decoded_probs

def entrenar_circuito(ansatz, param_count, data=train_data, validation=eval_data):
    fidelity_qnode, _, _ = crear_qnodes(ansatz)
    weights = pnp.array(rng.uniform(0, 2*np.pi, (N_LAYERS, TOTAL_QUBITS, param_count)), requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=LEARNING_RATE)
    history = []
    for _ in range(N_EPOCHS):
        order = rng.permutation(len(data))
        for start in range(0, len(data), BATCH_SIZE):
            batch = data[order[start:start+BATCH_SIZE]]
            def loss(w):
                return pnp.mean(pnp.stack([1.0 - fidelity_qnode(x, w)[0] for x in batch]))
            weights, _ = opt.step_and_cost(loss, weights)
        history.append(float(np.mean([fidelity_qnode(x, weights)[0] for x in validation])))
    return weights, history

results = {}
for name, (ansatz, param_count) in ANSATZE.items():
    print(f'Entrenando {name}...')
    results[name] = dict(zip(('weights', 'history'), entrenar_circuito(ansatz, param_count)))
print('Entrenamiento completado.')

## Dibujos y curvas de convergencia

In [ ]:
for name, (ansatz, _) in ANSATZE.items():
    dev_draw = qml.device('default.qubit', wires=TOTAL_QUBITS)
    @qml.qnode(dev_draw)
    def draw_qnode(weights):
        qml.AmplitudeEmbedding(train_data[0], wires=WIRES, normalize=False)
        ansatz(weights, WIRES)
        return qml.probs(wires=trash_wires)
    print(f'\n{name}')
    try:
        display(qml.draw_mpl(draw_qnode, decimals=1)(results[name]['weights']))
    except Exception as exc:
        print(qml.draw(draw_qnode, decimals=1)(results[name]['weights']))
        print(f'(Dibujo matplotlib no disponible: {exc})')

plt.figure(figsize=(12, 5))
for name, result in results.items():
    plt.plot(np.arange(1, N_EPOCHS + 1), result['history'], marker='o', label=name)
plt.xlabel('Época'); plt.ylabel('Fidelidad de validación'); plt.ylim(0, 1.05)
plt.title('Convergencia: fidelidad del trash'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## Reconstrucción y distribución de errores

Para evitar una reconstrucción trivial `U U†`, se conserva el estado después del encoder, se ponen a cero las amplitudes cuyo trash no es `|0...0>` y se renormaliza. Luego se aplica `U†`; el MSE compara las probabilidades del estado original y reconstruido.

In [ ]:
def proyectar_trash_a_cero(encoded):
    tensor = np.asarray(encoded).reshape([2] * TOTAL_QUBITS)
    moved = np.moveaxis(tensor, trash_wires, range(TOTAL_QUBITS - TRASH_QUBITS, TOTAL_QUBITS))
    projected = np.zeros_like(moved)
    keep = (slice(None),) * (TOTAL_QUBITS - TRASH_QUBITS) + (0,) * TRASH_QUBITS
    projected[keep] = moved[keep]
    projected = projected.reshape(-1)
    norm = np.linalg.norm(projected)
    return projected / norm if norm > 1e-12 else projected

def evaluar_reconstruccion(ansatz, weights, data=eval_data):
    fidelity_qnode, encoded_state, decoded_probs = crear_qnodes(ansatz)
    fidelidades, mses = [], []
    for sample in data:
        projected = proyectar_trash_a_cero(encoded_state(sample, weights))
        reconstructed = np.asarray(decoded_probs(projected, weights))
        fidelidades.append(float(fidelity_qnode(sample, weights)[0]))
        mses.append(float(np.mean((np.asarray(sample)**2 - reconstructed)**2)))
    return np.asarray(fidelidades), np.asarray(mses)

# Anomalías sintéticas: perturbación espectral controlada para visualizar separación.
anomaly_data = eval_data.copy()
anomaly_data[:, 10:25] += rng.normal(0.35, 0.08, anomaly_data[:, 10:25].shape)
anomaly_data /= np.where(np.linalg.norm(anomaly_data, axis=1, keepdims=True) == 0, 1.0,
                         np.linalg.norm(anomaly_data, axis=1, keepdims=True))

summary = []
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
for axis, (name, (ansatz, _)) in zip(axes.flat, ANSATZE.items()):
    healthy_f, healthy_mse = evaluar_reconstruccion(ansatz, results[name]['weights'], eval_data)
    anomaly_f, anomaly_mse = evaluar_reconstruccion(ansatz, results[name]['weights'], anomaly_data)
    results[name].update(fidelities=healthy_f, mses=healthy_mse,
                         anomaly_fidelities=anomaly_f, anomaly_mses=anomaly_mse)
    summary.append({'circuit': name, **complexity.loc[name].to_dict(),
                    'fidelity_mean': healthy_f.mean(), 'mse_mean': healthy_mse.mean(),
                    'anomaly_mse_mean': anomaly_mse.mean()})
    axis.hist(1 - healthy_f, bins=8, alpha=0.65, color='#2ecc71', label='Sanos')
    axis.hist(1 - anomaly_f, bins=8, alpha=0.65, color='#e74c3c', label='Anomalías')
    axis.set_title(name, fontsize=9); axis.legend(fontsize=7)
for axis in axes[-1]: axis.set_xlabel('Error (1 - fidelidad)')
for axis in axes[:, 0]: axis.set_ylabel('Frecuencia')
fig.suptitle('Distribución de errores: sanos vs anomalías sintéticas', y=1.02)
plt.tight_layout(); plt.show()
summary_df = pd.DataFrame(summary).sort_values('mse_mean')
display(summary_df[['circuit', 'depth', 'width', 'parameters', 'one_qubit_gates',
                    'two_qubit_gates', 'fidelity_mean', 'mse_mean', 'anomaly_mse_mean']])

## Cómo explicar los resultados

- **Fidelidad alta:** el encoder concentra más amplitud en los qubits latentes y deja el trash cerca de `|0...0>`.
- **MSE bajo:** tras proyectar el trash y aplicar `U†`, las probabilidades reconstruidas se parecen a las originales.
- **Profundidad y puertas de dos qubits:** suelen aumentar la capacidad, pero también el coste de simulación y la sensibilidad al ruido.
- La comparación es justa porque todos usan los mismos qubits, datos, optimizador y épocas; `EfficientSU2` tiene tres parámetros por qubit y capa.